In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Dependencies
# ─────────────────────────────────────────────
import sys
!{sys.executable} -m pip install -q \
    "diffusers>=0.25.0" transformers accelerate xformers \
    scikit-image lpips clean-fid opencv-python-headless scikit-learn
print("Dependencies ready")

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Config & Paths
# Config C: SD Inpainting + ControlNet-Canny + IP-Adapter (surface/color)
# Config D fallback: text color prompt nếu IP-Adapter OOM
# ─────────────────────────────────────────────
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATASET_ROOT = Path("/kaggle/input/datasets/dangvy1507/occlusion")
SYNTH_DIR    = DATASET_ROOT / "synthetic_occ"
GT_DIR       = SYNTH_DIR / "x_gt"
OCC_DIR      = SYNTH_DIR / "x_occ"
MASK_DIR     = SYNTH_DIR / "masks"
META_CSV     = SYNTH_DIR / "metadata_synthetic_occ.csv"

OUT_DIR       = Path("/kaggle/working/controlnet_ip_sd")
PRED_DIR      = OUT_DIR / "x_hat"
EVAL_GT_DIR   = OUT_DIR / "eval_gt_200"
EVAL_PRED_DIR = OUT_DIR / "eval_pred_200"
REPORT_DIR    = OUT_DIR / "reports"
for d in [PRED_DIR, EVAL_GT_DIR, EVAL_PRED_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

SD_MODEL_ID  = "runwayml/stable-diffusion-inpainting"
CN_MODEL_ID  = "lllyasviel/control_v11p_sd15_canny"
IP_REPO      = "h94/IP-Adapter"
IP_SUBFOLDER = "models"
IP_WEIGHT    = "ip-adapter-plus_sd15.bin"

PROMPT     = "a car, realistic, high quality, detailed"
NEG_PROMPT = "blurry, distorted, artifacts"
NUM_STEPS  = 20
GUIDANCE   = 7.5
TEST_SIZE  = 200

CN_SCALE_GRID = [0.25, 0.40, 0.55]
IP_SCALE_GRID = [0.3, 0.5, 0.7]
GRID_SAMPLE   = 20

CANNY_LOW      = 80
CANNY_HIGH     = 150
MASK_DILATE_PX = 5

meta = pd.read_csv(META_CSV)
print(f"Meta rows  : {len(meta):,}")
print(f"Device     : {DEVICE}")
print(f"SD model   : {SD_MODEL_ID}")
print(f"ControlNet : {CN_MODEL_ID}")
print(f"IP-Adapter : {IP_REPO}/{IP_SUBFOLDER}/{IP_WEIGHT}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Load SD Inpainting + ControlNet + IP-Adapter
# Paper IP-Adapter (2308.06721): decoupled cross-attention —
#   text features và image features đi qua cross-attention riêng biệt,
#   chỉ cần train thêm W'_k, W'_v cho mỗi layer (~22M params).
# IP-Adapter Plus dùng fine-grained features (grid features từ CLIP ViT-H/14)
#   → bắt được chi tiết surface/color tốt hơn global embedding.
# ─────────────────────────────────────────────
import cv2
from PIL import Image
from diffusers import (
    StableDiffusionControlNetInpaintPipeline,
    ControlNetModel,
    DPMSolverMultistepScheduler,
)

print("Loading ControlNet ...")
controlnet = ControlNetModel.from_pretrained(CN_MODEL_ID, torch_dtype=DTYPE)

print("Loading SD Inpainting + ControlNet pipeline ...")
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    SD_MODEL_ID,
    controlnet=controlnet,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEVICE)

xformers_ok = False
try:
    pipe.enable_xformers_memory_efficient_attention()
    xformers_ok = True
    print("xFormers: enabled")
except Exception:
    pipe.enable_attention_slicing("auto")
    print("xFormers: fallback to attention slicing")
pipe.vae.enable_slicing()

USE_IP_ADAPTER = False
try:
    print("Loading IP-Adapter Plus ...")
    pipe.load_ip_adapter(IP_REPO, subfolder=IP_SUBFOLDER, weight_name=IP_WEIGHT)
    USE_IP_ADAPTER = True
    print("IP-Adapter loaded successfully (Config C)")
except Exception as e:
    print(f"IP-Adapter load failed: {e}")
    print("Falling back to Config D (text color prompt)")

config_label = "C_SD_CN_IP" if USE_IP_ADAPTER else "D_SD_CN_TextColor"
print(f"Pipeline ready | Scheduler: {pipe.scheduler.__class__.__name__} | xFormers: {xformers_ok}")
print(f"Active config: {config_label}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Helper functions
#   extract_canny_masked : Canny từ x_occ, xoá cạnh trong vùng mask
#   extract_visible_patch: P = visible region, fill masked area with mean color
#   get_color_prompt     : Config D fallback — dominant color → text prompt
#   inpaint_cn_ip        : inference wrapper (Config C hoặc D tự động)
# ─────────────────────────────────────────────

def extract_canny_masked(
    image_pil: Image.Image,
    mask_pil: Image.Image,
    low: int = CANNY_LOW,
    high: int = CANNY_HIGH,
    dilate_px: int = MASK_DILATE_PX,
) -> Image.Image:
    """Trích Canny từ x_occ, xoá cạnh trong vùng mask + dilate margin."""
    img = np.array(image_pil.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, low, high)
    mask_np = np.array(mask_pil.convert("L"))
    if dilate_px > 0:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (dilate_px * 2 + 1, dilate_px * 2 + 1)
        )
        mask_np = cv2.dilate(mask_np, kernel, iterations=1)
    edges[mask_np > 127] = 0
    return Image.fromarray(np.stack([edges] * 3, axis=2))


def extract_visible_patch(image_pil: Image.Image, mask_pil: Image.Image) -> Image.Image:
    """
    P = x_occ masked to visible region only.
    Occluded area filled with mean color of visible pixels →
    CLIP encoder nhận ảnh sạch, không bị nhiễu bởi occluder.
    """
    img = np.array(image_pil.convert("RGB"))
    msk = np.array(mask_pil.convert("L"))
    visible = msk < 128
    if visible.sum() > 0:
        mean_color = img[visible].mean(axis=0).astype(np.uint8)
    else:
        mean_color = np.array([128, 128, 128], dtype=np.uint8)
    patch = img.copy()
    patch[~visible] = mean_color
    return Image.fromarray(patch)


def get_color_prompt(image_pil: Image.Image, mask_pil: Image.Image) -> str:
    """Config D fallback: trích dominant color → descriptive prompt thay IP-Adapter."""
    img = np.array(image_pil.convert("RGB"))
    msk = np.array(mask_pil.convert("L"))
    visible_px = img[msk < 128]
    if len(visible_px) < 100:
        return PROMPT
    from sklearn.cluster import KMeans
    km = KMeans(n_clusters=3, random_state=SEED, n_init=10)
    km.fit(visible_px.reshape(-1, 3))
    labels, counts = np.unique(km.labels_, return_counts=True)
    dominant_rgb = km.cluster_centers_[labels[np.argmax(counts)]].astype(np.uint8)
    hsv = cv2.cvtColor(np.uint8([[dominant_rgb]]), cv2.COLOR_RGB2HSV)[0, 0]
    h, s, v = int(hsv[0]), int(hsv[1]), int(hsv[2])
    if v < 50:
        name = "black"
    elif s < 30 and v > 200:
        name = "white"
    elif s < 40:
        name = "silver" if v > 140 else "gray"
    elif h < 10 or h > 170:
        name = "red"
    elif h < 25:
        name = "orange"
    elif h < 35:
        name = "yellow"
    elif h < 85:
        name = "green"
    elif h < 130:
        name = "blue"
    elif h < 160:
        name = "purple"
    else:
        name = "red"
    return f"a {name} car, realistic, high quality, detailed, consistent paint color"


def inpaint_cn_ip(
    image: Image.Image,
    mask: Image.Image,
    prompt: str,
    cn_scale: float = 0.4,
    ip_scale: float = 0.5,
    seed: int = SEED,
) -> Image.Image:
    """Config C/D inference: ControlNet-Canny + IP-Adapter (hoặc text color fallback)."""
    canny = extract_canny_masked(image, mask)
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    call_kwargs = dict(
        prompt=prompt,
        negative_prompt=NEG_PROMPT,
        image=image,
        control_image=canny,
        mask_image=mask,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE,
        controlnet_conditioning_scale=cn_scale,
        generator=generator,
    )

    if USE_IP_ADAPTER:
        visible = extract_visible_patch(image, mask)
        pipe.set_ip_adapter_scale(ip_scale)
        call_kwargs["ip_adapter_image"] = visible
    else:
        call_kwargs["prompt"] = get_color_prompt(image, mask)

    with torch.autocast("cuda"):
        out = pipe(**call_kwargs).images[0]
    torch.cuda.empty_cache()
    return out

print(f"Helper functions defined | Config: {config_label}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Visualize: x_occ / Mask / Canny / Visible patch / x_gt
# ─────────────────────────────────────────────
import matplotlib.pyplot as plt

meta_sample = meta.sample(4, random_state=SEED).reset_index(drop=True)
n_cols = 5 if USE_IP_ADAPTER else 4
fig, axes = plt.subplots(4, n_cols, figsize=(4 * n_cols, 14))
for i, row in meta_sample.iterrows():
    occ = Image.open(OCC_DIR / row["x_occ"]).convert("RGB").resize((512, 512))
    msk = Image.open(MASK_DIR / f"{row['stem']}.png").convert("L").resize((512, 512))
    gt  = Image.open(GT_DIR  / row["x_gt"]).convert("RGB").resize((512, 512))
    can = extract_canny_masked(occ, msk)
    col = 0
    if i == 0: axes[i, col].set_title("x_occ")
    axes[i, col].imshow(occ); col += 1
    if i == 0: axes[i, col].set_title("Mask")
    axes[i, col].imshow(msk, cmap="gray"); col += 1
    if i == 0: axes[i, col].set_title("Canny (masked)")
    axes[i, col].imshow(can); col += 1
    if USE_IP_ADAPTER:
        vis = extract_visible_patch(occ, msk)
        if i == 0: axes[i, col].set_title("Visible patch (P)")
        axes[i, col].imshow(vis); col += 1
    if i == 0: axes[i, col].set_title("x_gt")
    axes[i, col].imshow(gt)
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Grid search cn_scale × ip_scale
# 2D grid: cn_scale × ip_scale (Config D: chỉ cn_scale)
# ─────────────────────────────────────────────
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim_fn
import lpips as lpips_lib

lpips_model = lpips_lib.LPIPS(net="alex").to(DEVICE)
lpips_model.eval()


def masked_ssim(gt, pred, mask01):
    gt_f = gt.astype(np.float32) / 255.0
    pr_f = pred.astype(np.float32) / 255.0
    _, ssim_map = ssim_fn(gt_f, pr_f, channel_axis=2, data_range=1.0, full=True)
    m = mask01.astype(bool)
    return float(ssim_map[m].mean()) if m.sum() > 0 else float(ssim_map.mean())


def masked_lpips(gt, pred, mask01):
    m = mask01.astype(np.float32)[..., None]
    gt_m = (gt.astype(np.float32) * m).astype(np.uint8)
    pr_m = (pred.astype(np.float32) * m).astype(np.uint8)
    gt_t = torch.from_numpy(gt_m).permute(2, 0, 1).unsqueeze(0).float() / 127.5 - 1.0
    pr_t = torch.from_numpy(pr_m).permute(2, 0, 1).unsqueeze(0).float() / 127.5 - 1.0
    with torch.no_grad():
        return float(lpips_model(gt_t.to(DEVICE), pr_t.to(DEVICE)).item())


grid_sample = meta.sample(GRID_SAMPLE, random_state=SEED).reset_index(drop=True)
grid_res = []
ip_values = IP_SCALE_GRID if USE_IP_ADAPTER else [0.0]

for cn_s in CN_SCALE_GRID:
    for ip_s in ip_values:
        ss, lp = [], []
        tag = f"cn={cn_s},ip={ip_s}" if USE_IP_ADAPTER else f"cn={cn_s}"
        for _, r in tqdm(grid_sample.iterrows(), total=len(grid_sample), desc=tag):
            occ  = Image.open(OCC_DIR  / r["x_occ"]).convert("RGB").resize((512, 512))
            msk  = Image.open(MASK_DIR / f"{r['stem']}.png").convert("L").resize((512, 512))
            gt   = Image.open(GT_DIR   / r["x_gt"]).convert("RGB").resize((512, 512))
            pred = inpaint_cn_ip(occ, msk, PROMPT, cn_scale=cn_s, ip_scale=ip_s, seed=SEED)
            gt_np = np.array(gt)
            pr_np = np.array(pred)
            mk_np = cv2.imread(str(MASK_DIR / f"{r['stem']}.png"), cv2.IMREAD_GRAYSCALE)
            mk01  = (cv2.resize(mk_np, (512, 512)) > 127).astype(np.uint8)
            ss.append(masked_ssim(gt_np, pr_np, mk01))
            lp.append(masked_lpips(gt_np, pr_np, mk01))
        grid_res.append({
            "cn_scale": cn_s, "ip_scale": ip_s,
            "ssim_mean": float(np.mean(ss)), "lpips_mean": float(np.mean(lp)),
        })
        print(f"  {tag}  SSIM={np.mean(ss):.4f} | LPIPS={np.mean(lp):.4f}")

grid_df = pd.DataFrame(grid_res)
best_idx = grid_df["lpips_mean"].idxmin()
BEST_CN = float(grid_df.loc[best_idx, "cn_scale"])
BEST_IP = float(grid_df.loc[best_idx, "ip_scale"])
print(f"\nBest: cn_scale={BEST_CN}, ip_scale={BEST_IP}")
grid_df.to_csv(REPORT_DIR / "grid_search.csv", index=False)
print(grid_df.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Full inference 200 images + Evaluate (SSIM, LPIPS, FID)
# ─────────────────────────────────────────────
import time
from cleanfid import fid

test_df = meta.sample(TEST_SIZE, random_state=SEED).reset_index(drop=True)
pred_records = []
t0 = time.time()

for idx, r in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    occ = Image.open(OCC_DIR  / r["x_occ"]).convert("RGB").resize((512, 512))
    msk = Image.open(MASK_DIR / f"{r['stem']}.png").convert("L").resize((512, 512))
    gt  = Image.open(GT_DIR   / r["x_gt"]).convert("RGB").resize((512, 512))

    pred = inpaint_cn_ip(occ, msk, PROMPT, cn_scale=BEST_CN, ip_scale=BEST_IP, seed=SEED)

    pred_name = f"{r['stem']}.png"
    pred.save(PRED_DIR / pred_name)
    gt.save(EVAL_GT_DIR / pred_name)
    pred.save(EVAL_PRED_DIR / pred_name)
    pred_records.append({
        "stem": r["stem"], "x_gt": r["x_gt"], "x_occ": r["x_occ"],
        "occlusion_ratio": r["occlusion_ratio"], "pred": pred_name,
    })

infer_elapsed = time.time() - t0
pred_df = pd.DataFrame(pred_records)
print(f"Inference done: {len(pred_df)} images in {infer_elapsed/60:.1f} min")
print(f"Config: {config_label} | cn_scale={BEST_CN} | ip_scale={BEST_IP}")

# ── Compute per-image metrics ──
metric_rows = []
for _, r in tqdm(pred_df.iterrows(), total=len(pred_df), desc="Metrics"):
    gt = cv2.cvtColor(cv2.imread(str(GT_DIR / r["x_gt"])), cv2.COLOR_BGR2RGB)
    pr = cv2.cvtColor(cv2.imread(str(PRED_DIR / r["pred"])), cv2.COLOR_BGR2RGB)
    gt = cv2.resize(gt, (512, 512))
    pr = cv2.resize(pr, (512, 512))
    mk = cv2.imread(str(MASK_DIR / f"{r['stem']}.png"), cv2.IMREAD_GRAYSCALE)
    mk = cv2.resize(mk, (512, 512))
    mk01 = (mk > 127).astype(np.uint8)
    metric_rows.append({
        "stem": r["stem"],
        "occlusion_ratio": r["occlusion_ratio"],
        "ssim": masked_ssim(gt, pr, mk01),
        "lpips": masked_lpips(gt, pr, mk01),
    })

metric_df = pd.DataFrame(metric_rows)
fid_value = fid.compute_fid(str(EVAL_GT_DIR), str(EVAL_PRED_DIR), mode="clean")

summary = {
    "config": config_label,
    "model": f"{SD_MODEL_ID} + {CN_MODEL_ID} + {IP_WEIGHT}",
    "scheduler": pipe.scheduler.__class__.__name__,
    "num_test_images": int(len(metric_df)),
    "prompt": PROMPT,
    "negative_prompt": NEG_PROMPT,
    "steps": int(NUM_STEPS),
    "guidance": float(GUIDANCE),
    "cn_scale": float(BEST_CN),
    "ip_scale": float(BEST_IP),
    "ssim_mean": float(metric_df["ssim"].mean()),
    "lpips_mean": float(metric_df["lpips"].mean()),
    "fid": float(fid_value),
    "inference_minutes": float(infer_elapsed / 60.0),
}

metric_df.to_csv(REPORT_DIR / f"{config_label}_metrics_per_image.csv", index=False)
pd.DataFrame([summary]).to_csv(REPORT_DIR / f"{config_label}_summary.csv", index=False)

print("=" * 60)
print(f"Config     : {config_label}")
print(f"SSIM  mean : {summary['ssim_mean']:.4f}   (baseline A: 0.4276)")
print(f"LPIPS mean : {summary['lpips_mean']:.4f}   (baseline A: 0.1011)")
print(f"FID        : {summary['fid']:.2f}    (baseline A: 35.67)")
print(f"cn_scale={BEST_CN}, ip_scale={BEST_IP}")
print("=" * 60)

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Analysis: best/worst cases
# ─────────────────────────────────────────────
import matplotlib.pyplot as plt

analysis_df = metric_df.copy()
analysis_df["score"] = analysis_df["ssim"] - analysis_df["lpips"]
best20  = analysis_df.sort_values("score", ascending=False).head(20)
worst20 = analysis_df.sort_values("score", ascending=True).head(20)
best20.to_csv(REPORT_DIR  / "best20_cases.csv",  index=False)
worst20.to_csv(REPORT_DIR / "worst20_cases.csv", index=False)


def show_cases(case_df: pd.DataFrame, title: str, n_show: int = 10):
    n = min(n_show, len(case_df))
    fig, axes = plt.subplots(n, 4, figsize=(14, 3 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)
    for i, (_, row) in enumerate(case_df.head(n).iterrows()):
        stem = row["stem"]
        gt   = Image.open(EVAL_GT_DIR   / f"{stem}.png").convert("RGB")
        pr   = Image.open(EVAL_PRED_DIR / f"{stem}.png").convert("RGB")
        rr   = pred_df[pred_df["stem"] == stem].iloc[0]
        occ  = Image.open(OCC_DIR  / rr["x_occ"]).convert("RGB")
        mask = Image.open(MASK_DIR / f"{stem}.png").convert("L")
        axes[i, 0].imshow(gt);                axes[i, 0].set_title("x_gt")
        axes[i, 1].imshow(occ);               axes[i, 1].set_title("x_occ")
        axes[i, 2].imshow(mask, cmap="gray"); axes[i, 2].set_title("M")
        axes[i, 3].imshow(pr);                axes[i, 3].set_title(
            f"x_hat\nSSIM={row['ssim']:.3f} LPIPS={row['lpips']:.3f}"
        )
        for j in range(4):
            axes[i, j].axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


show_cases(best20,  f"Best 10 ({config_label})",  n_show=10)
show_cases(worst20, f"Worst 10 ({config_label})", n_show=10)

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — VRAM check
# ─────────────────────────────────────────────
import subprocess

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    r = test_df.iloc[0]
    img_occ  = Image.open(OCC_DIR  / r["x_occ"]).convert("RGB").resize((512, 512))
    img_mask = Image.open(MASK_DIR / f"{r['stem']}.png").convert("L").resize((512, 512))
    _ = inpaint_cn_ip(img_occ, img_mask, PROMPT, cn_scale=BEST_CN, ip_scale=BEST_IP, seed=SEED)
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    print(f"Peak VRAM (1 image): {peak_mb:.0f} MiB ({peak_mb/1024:.2f} GiB)")
    print(f"xFormers: {xformers_ok} | IP-Adapter: {USE_IP_ADAPTER}")
    try:
        smi = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free",
             "--format=csv,noheader,nounits"],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
        print("nvidia-smi:\n" + smi)
    except Exception:
        print("nvidia-smi not available")
else:
    print("No CUDA GPU")